In [1]:
import pandas as pd
import numpy as np
from pathlib import Path 
from scipy.stats import norm

In [2]:
forecast_14d = pd.read_parquet(
    "notebooks/data/processed/forecast/future_forecast_14d.parquet"
).rename(columns={
    "Store_ID": "Store ID",
    "Product_ID": "Product ID"
})

forecast_14d.head()

,Date,Store ID,Product ID,Forecast Demand
0,2023-11-01,S001,P0001,61.947955
1,2023-11-01,S001,P0002,103.120745
2,2023-11-01,S001,P0003,128.367267
3,2023-11-01,S001,P0004,85.704218
4,2023-11-01,S001,P0005,208.519785


In [3]:
print("Toplam satır:", len(forecast_14d))
print("Farklı gün sayısı:", forecast_14d["Date"].nunique())
print(forecast_14d.groupby("Date").size())

Toplam satır: 1400
Farklı gün sayısı: 14
Date
2023-11-01    100
2023-11-02    100
2023-11-03    100
2023-11-04    100
2023-11-05    100
2023-11-06    100
2023-11-07    100
2023-11-08    100
2023-11-09    100
2023-11-10    100
2023-11-11    100
2023-11-12    100
2023-11-13    100
2023-11-14    100
dtype: int64


In [4]:
display(
    forecast_14d[
        (forecast_14d["Store ID"] == "S001") &
        (forecast_14d["Product ID"] == "P0001")
    ].sort_values("Date")
)

,Date,Store ID,Product ID,Forecast Demand
0,2023-11-01,S001,P0001,61.947955
100,2023-11-02,S001,P0001,32.512504
200,2023-11-03,S001,P0001,31.363328
300,2023-11-04,S001,P0001,31.729929
400,2023-11-05,S001,P0001,31.729929
500,2023-11-06,S001,P0001,31.568763
600,2023-11-07,S001,P0001,31.682878
700,2023-11-08,S001,P0001,31.315355
800,2023-11-09,S001,P0001,31.315355
900,2023-11-10,S001,P0001,31.291548


In [5]:
df_history = pd.concat([
    pd.read_parquet("notebooks/data/processed/main/train.parquet"),
    pd.read_parquet("notebooks/data/processed/main/validation.parquet"),
    pd.read_parquet("notebooks/data/processed/main/test.parquet"),
], ignore_index=True)

df_history.head()

,Date,Store ID,Product ID,Inventory Level,Units Sold,Units Ordered,Price,Discount,Holiday/Promotion,Year,...,Category_Toys,Region_North,Region_South,Region_West,Weather Condition_Rainy,Weather Condition_Snowy,Weather Condition_Sunny,Seasonality_NEW_Spring,Seasonality_NEW_Summer,Seasonality_NEW_Winter
0,2022-01-01,S001,P0001,231,127,55,33.50,20,0,2022,...,False,True,False,False,True,False,False,0,0,1
1,2022-01-02,S001,P0001,116,81,104,27.95,10,0,2022,...,False,False,False,True,False,False,False,0,0,1
2,2022-01-03,S001,P0001,154,5,189,62.70,20,0,2022,...,False,False,False,True,True,False,False,0,0,1
3,2022-01-04,S001,P0001,85,58,193,77.88,15,1,2022,...,False,False,True,False,False,False,False,0,0,1
4,2022-01-05,S001,P0001,238,147,37,28.46,20,1,2022,...,False,False,True,False,False,False,True,0,0,1


In [6]:
latest_inventory = (
    df_history
    .sort_values("Date")
    .groupby(["Store ID", "Product ID"])["Inventory Level"]
    .last()
    .reset_index()
)

latest_inventory.head()

,Store ID,Product ID,Inventory Level
0,S001,P0001,223
1,S001,P0002,217
2,S001,P0003,69
3,S001,P0004,338
4,S001,P0005,471


In [7]:
inventory_forecast = forecast_14d.merge(
    latest_inventory,
    on=["Store ID", "Product ID"],
    how="left"
)

inventory_forecast.head()

,Date,Store ID,Product ID,Forecast Demand,Inventory Level
0,2023-11-01,S001,P0001,61.947955,223
1,2023-11-01,S001,P0002,103.120745,217
2,2023-11-01,S001,P0003,128.367267,69
3,2023-11-01,S001,P0004,85.704218,338
4,2023-11-01,S001,P0005,208.519785,471


In [8]:
inventory_forecast = inventory_forecast.sort_values(
    ["Store ID", "Product ID", "Date"]
).copy()

inventory_forecast["Cumulative Forecast"] = (
    inventory_forecast
    .groupby(["Store ID", "Product ID"])["Forecast Demand"]
    .cumsum()
)

inventory_forecast["Projected Inventory"] = (
    inventory_forecast["Inventory Level"]
    - inventory_forecast["Cumulative Forecast"]
)

inventory_forecast.head()


,Date,Store ID,Product ID,Forecast Demand,Inventory Level,Cumulative Forecast,Projected Inventory
0,2023-11-01,S001,P0001,61.947955,223,61.947955,161.052045
100,2023-11-02,S001,P0001,32.512504,223,94.460459,128.539541
200,2023-11-03,S001,P0001,31.363328,223,125.823788,97.176212
300,2023-11-04,S001,P0001,31.729929,223,157.553717,65.446283
400,2023-11-05,S001,P0001,31.729929,223,189.283645,33.716355


In [9]:
inventory_forecast["Stockout Risk"] = np.where(
    inventory_forecast["Projected Inventory"] <= 0,
    "HIGH",
    "LOW")

In [18]:
risk_summary = (
    inventory_forecast
    .groupby(["Store ID", "Product ID"])
    .agg(
        Current_Inventory=("Inventory Level", "first"),
        Min_Projected_Inventory=("Projected Inventory", "min"),
        Forecast_14D=("Forecast Demand", "sum"),
    )
    .reset_index()
)

# Forecast_7D icin ayri bir hesap: sadece ilk 7 gunun toplami
forecast_7d = (
    inventory_forecast
    .sort_values(["Store ID", "Product ID", "Date"])
    .groupby(["Store ID", "Product ID"])["Forecast Demand"]
    .apply(lambda x: x.iloc[:7].sum())
    .reset_index(name="Forecast_7D")
)

risk_summary = risk_summary.merge(forecast_7d, on=["Store ID", "Product ID"], how="left")

risk_summary.head()

,Store ID,Product ID,Current_Inventory,Min_Projected_Inventory,Forecast_14D,Forecast_7D
0,S001,P0001,223,-249.064358,472.064358,252.535287
1,S001,P0002,217,-354.772979,571.772979,314.459390
2,S001,P0003,69,-521.684107,590.684107,350.523940
3,S001,P0004,338,-172.752444,510.752444,288.853863
4,S001,P0005,471,-254.117065,725.117065,502.888410


In [20]:
risk_summary["Stockout Risk"] = np.where(
    risk_summary["Min_Projected_Inventory"] <= 0,
    "HIGH",
    "LOW")

In [22]:
risk_summary["Avg_Daily_Forecast"] = (risk_summary["Forecast_14D"] / 14)
risk_summary["Days_of_Cover"] = (risk_summary["Current_Inventory"]
    / risk_summary["Avg_Daily_Forecast"].replace(0, np.nan))

In [24]:
risk_summary[
    [
        "Store ID",
        "Product ID",
        "Current_Inventory",
        "Avg_Daily_Forecast",
        "Days_of_Cover"]].head(20)

,Store ID,Product ID,Current_Inventory,Avg_Daily_Forecast,Days_of_Cover
0,S001,P0001,223,33.718883,6.613505
1,S001,P0002,217,40.840927,5.313298
2,S001,P0003,69,42.191722,1.635392
3,S001,P0004,338,36.482317,9.264762
4,S001,P0005,471,51.794076,9.093704
5,S001,P0006,305,31.499185,9.682790
6,S001,P0007,256,32.903648,7.780292
7,S001,P0008,315,34.252561,9.196393
8,S001,P0009,167,44.201392,3.778162
9,S001,P0010,167,31.610973,5.282976


In [26]:
def classify_stock_risk(days):

    if pd.isna(days):
        return "NO DEMAND"

    elif days < 3:
        return "HIGH"

    elif days < 7:
        return "MEDIUM"

    elif days <= 14:
        return "LOW"

    else:
        return "OVERSTOCK"

In [28]:
risk_summary["Risk Level"] = (
    risk_summary["Days_of_Cover"]
    .apply(classify_stock_risk))

In [30]:
risk_summary[
    [
        "Store ID",
        "Product ID",
        "Current_Inventory",
        "Forecast_7D",
        "Forecast_14D",
        "Days_of_Cover",
        "Stockout Risk",
        "Risk Level"]].head(20)

,Store ID,Product ID,Current_Inventory,Forecast_7D,Forecast_14D,Days_of_Cover,Stockout Risk,Risk Level
0,S001,P0001,223,252.535287,472.064358,6.613505,HIGH,MEDIUM
1,S001,P0002,217,314.459390,571.772979,5.313298,HIGH,MEDIUM
2,S001,P0003,69,350.523940,590.684107,1.635392,HIGH,HIGH
3,S001,P0004,338,288.853863,510.752444,9.264762,HIGH,LOW
4,S001,P0005,471,502.888410,725.117065,9.093704,HIGH,LOW
5,S001,P0006,305,226.757705,440.988595,9.682790,HIGH,LOW
6,S001,P0007,256,216.031147,460.651066,7.780292,HIGH,LOW
7,S001,P0008,315,265.131052,479.535856,9.196393,HIGH,LOW
8,S001,P0009,167,404.268997,618.819492,3.778162,HIGH,MEDIUM
9,S001,P0010,167,217.521625,442.553625,5.282976,HIGH,MEDIUM


In [32]:
risk_summary["Risk Level"].value_counts()

Risk Level
LOW          48
MEDIUM       35
HIGH         12
OVERSTOCK     5
Name: count, dtype: int64

In [34]:
LEAD_TIME_DAYS = 3
SERVICE_LEVEL_Z = 1.65

In [36]:
demand_stats = (
    df_history
    .groupby(["Store ID", "Product ID"])["Units Sold"]
    .agg(
        Demand_Mean="mean",
        Demand_Std="std"
    )
    .reset_index()
)

demand_stats.head()

,Store ID,Product ID,Demand_Mean,Demand_Std
0,S001,P0001,137.306430,108.187859
1,S001,P0002,131.038304,104.313220
2,S001,P0003,141.440492,107.211225
3,S001,P0004,140.904241,113.644117
4,S001,P0005,133.129959,107.368873


In [38]:
risk_summary = risk_summary.merge(
    demand_stats,
    on=["Store ID", "Product ID"],
    how="left"
)

In [40]:
risk_summary["Safety_Stock"] = (
    SERVICE_LEVEL_Z
    * risk_summary["Demand_Std"]
    * np.sqrt(LEAD_TIME_DAYS)
)

In [42]:
risk_summary["Safety_Stock"] = (
    risk_summary["Safety_Stock"]
    .fillna(0)
    .clip(lower=0)
)

In [44]:
lead_time_forecast = (
    forecast_14d
    .sort_values(["Store ID", "Product ID", "Date"])
    .groupby(["Store ID", "Product ID"])
    ["Forecast Demand"]
    .apply(lambda x: x.iloc[:LEAD_TIME_DAYS].sum())
    .reset_index(name="Lead_Time_Demand")
)

In [46]:
risk_summary = risk_summary.merge(
    lead_time_forecast,
    on=["Store ID", "Product ID"],
    how="left"
)

In [48]:
risk_summary["Target_Stock"] = (
    risk_summary["Lead_Time_Demand"]
    + risk_summary["Safety_Stock"]
)

In [50]:
risk_summary["Recommended_Order_Qty"] = (
    risk_summary["Target_Stock"]
    - risk_summary["Current_Inventory"]
)

In [ ]:
risk_summary["Recommended_Order_Qty"] = (
    risk_summary["Recommended_Order_Qty"]
    .clip(lower=0)
)

In [52]:
risk_summary["Recommended_Order_Qty"] = (
    np.ceil(
        risk_summary["Recommended_Order_Qty"]
    ).astype(int)
)

In [54]:
final_risk_analysis = risk_summary[
    [
        "Store ID",
        "Product ID",
        "Current_Inventory",
        "Forecast_7D",
        "Forecast_14D",
        "Avg_Daily_Forecast",
        "Days_of_Cover",
        "Demand_Mean",
        "Demand_Std",
        "Lead_Time_Demand",
        "Safety_Stock",
        "Target_Stock",
        "Min_Projected_Inventory",
        "Stockout Risk",
        "Risk Level",
        "Recommended_Order_Qty"
    ]
].copy()

In [56]:
final_risk_analysis.head(20)

,Store ID,Product ID,Current_Inventory,Forecast_7D,Forecast_14D,Avg_Daily_Forecast,Days_of_Cover,Demand_Mean,Demand_Std,Lead_Time_Demand,Safety_Stock,Target_Stock,Min_Projected_Inventory,Stockout Risk,Risk Level,Recommended_Order_Qty
0,S001,P0001,223,252.535287,472.064358,33.718883,6.613505,137.306430,108.187859,125.823788,309.188333,435.012121,-249.064358,HIGH,MEDIUM,213
1,S001,P0002,217,314.459390,571.772979,40.840927,5.313298,131.038304,104.313220,192.039919,298.115065,490.154984,-354.772979,HIGH,MEDIUM,274
2,S001,P0003,69,350.523940,590.684107,42.191722,1.635392,141.440492,107.211225,225.921149,306.397227,532.318376,-521.684107,HIGH,HIGH,464
3,S001,P0004,338,288.853863,510.752444,36.482317,9.264762,140.904241,113.644117,162.272677,324.781685,487.054362,-172.752444,HIGH,LOW,150
4,S001,P0005,471,502.888410,725.117065,51.794076,9.093704,133.129959,107.368873,376.164260,306.847766,683.012026,-254.117065,HIGH,LOW,213
5,S001,P0006,305,226.757705,440.988595,31.499185,9.682790,130.673051,109.015261,102.104975,311.552953,413.657928,-135.988595,HIGH,LOW,109
6,S001,P0007,256,216.031147,460.651066,32.903648,7.780292,137.823529,111.198898,94.060984,317.793532,411.854515,-204.651066,HIGH,LOW,156
7,S001,P0008,315,265.131052,479.535856,34.252561,9.196393,137.132695,108.699045,139.944750,310.649243,450.593993,-164.535856,HIGH,LOW,136
8,S001,P0009,167,404.268997,618.819492,44.201392,3.778162,131.436389,107.086803,279.102021,306.041642,585.143663,-451.819492,HIGH,MEDIUM,419
9,S001,P0010,167,217.521625,442.553625,31.610973,5.282976,132.361149,105.916371,93.768498,302.696684,396.465182,-275.553625,HIGH,MEDIUM,230


In [58]:
OUTPUT_PATH = Path("outputs")

OUTPUT_PATH.mkdir(
    parents=True,
    exist_ok=True
)

final_risk_analysis.to_parquet(
    OUTPUT_PATH / "inventory_risk_analysis.parquet",
    index=False
)

print(
    "inventory_risk_analysis.parquet kaydedildi."
)

inventory_risk_analysis.parquet kaydedildi.


In [60]:
final_risk_analysis.to_csv(
    OUTPUT_PATH / "inventory_risk_analysis.csv",
    index=False
)

print("CSV kaydedildi.")

CSV kaydedildi.


In [62]:
print("===== SMARTSUPPLY INVENTORY SUMMARY =====")

print(
    "\nRisk dağılımı:"
)

print(
    final_risk_analysis["Risk Level"]
    .value_counts()
)

print(
    "\nStock-out riski yüksek ürün sayısı:",
    (
        final_risk_analysis["Stockout Risk"] == "HIGH"
    ).sum()
)

print(
    "\nToplam önerilen sipariş:",
    final_risk_analysis["Recommended_Order_Qty"].sum()
)

===== SMARTSUPPLY INVENTORY SUMMARY =====

Risk dağılımı:
Risk Level
LOW          48
MEDIUM       35
HIGH         12
OVERSTOCK     5
Name: count, dtype: int64

Stock-out riski yüksek ürün sayısı: 95

Toplam önerilen sipariş: 20712


In [64]:
top_orders = (
    final_risk_analysis
    .sort_values(
        "Recommended_Order_Qty",
        ascending=False
    )
    .head(10)
)

top_orders[
    [
        "Store ID",
        "Product ID",
        "Current_Inventory",
        "Forecast_14D",
        "Risk Level",
        "Recommended_Order_Qty"
    ]
]

,Store ID,Product ID,Current_Inventory,Forecast_14D,Risk Level,Recommended_Order_Qty
15,S001,P0016,74,620.905299,HIGH,516
78,S004,P0019,65,590.735149,HIGH,488
31,S002,P0012,193,684.497355,MEDIUM,473
86,S005,P0007,59,543.205033,HIGH,468
2,S001,P0003,69,590.684107,HIGH,464
26,S002,P0007,53,525.655222,HIGH,457
77,S004,P0018,83,579.912759,HIGH,454
82,S005,P0003,98,568.259526,HIGH,444
11,S001,P0012,213,687.369003,MEDIUM,429
8,S001,P0009,167,618.819492,MEDIUM,419


In [66]:
high_risk_products = (
    final_risk_analysis[
        final_risk_analysis["Risk Level"] == "HIGH"
    ]
    .sort_values(
        "Days_of_Cover"
    )
)

high_risk_products[
    [
        "Store ID",
        "Product ID",
        "Current_Inventory",
        "Days_of_Cover",
        "Forecast_14D",
        "Safety_Stock",
        "Recommended_Order_Qty"
    ]
].head(20)

,Store ID,Product ID,Current_Inventory,Days_of_Cover,Forecast_14D,Safety_Stock,Recommended_Order_Qty
26,S002,P0007,53,1.411572,525.655222,322.288399,457
86,S005,P0007,59,1.520604,543.205033,319.731397,468
78,S004,P0019,65,1.540453,590.735149,316.006965,488
2,S001,P0003,69,1.635392,590.684107,306.397227,464
15,S001,P0016,74,1.668531,620.905299,307.333770,516
28,S002,P0009,62,1.753421,495.032210,321.287926,417
74,S004,P0015,56,1.792412,437.399447,309.227556,348
77,S004,P0018,83,2.003750,579.912759,296.410459,454
82,S005,P0003,98,2.414390,568.259526,319.870864,444
71,S004,P0012,85,2.634577,451.685436,297.105607,317
